# 🚀 DAM-CMA: High-Performance Cloud Training
This notebook is designed to run on **Google Colab** using a **GPU (T4)**. It will train the main research model and all baselines on your custom 2,250-article dataset.

### **Instructions:**
1. **Runtime Type**: Go to `Runtime` -> `Change runtime type` -> Select `GPU (T4)`.
2. **Upload Data**: Click the Folder icon on the left sidebar and upload your `training_data.zip` file.
3. **Run All**: Click `Runtime` -> `Run all`.

## 1. Setup Environment

In [ ]:
# Clone the repository
!git clone https://github.com/harshala334/fyp.git
%cd fyp

# Install dependencies
!pip install transformers matplotlib opencv-python-headless pillow

# Unzip the data into the project folder
!unzip -o ../training_data.zip -d ./

import torch
print(f"GPU Available: {torch.cuda.is_available()}")

## 2. Initialize Models

In [ ]:
from src.data_loader import get_dataloader
from src.models.dam_cma import DAM_CMA_Model
from baselines.simple_fusion import SimpleFusionModel
from baselines.unimodal import TextOnlyBaseline, ImageOnlyBaseline
from src.train_utils import train_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load Data
train_loader = get_dataloader("data/master_data.jsonl", batch_size=16, shuffle=True)

# Models
main_model = DAM_CMA_Model(num_domains=2).to(device)
fusion_baseline = SimpleFusionModel().to(device)
text_baseline = TextOnlyBaseline().to(device)
image_baseline = ImageOnlyBaseline().to(device)

## 3. Start Training Phase
This will take approximately 10-15 minutes on a Tesla T4 GPU.

In [ ]:
print("--- Training Main Model (DAM-CMA) ---")
train_model(main_model, train_loader, epochs=10, is_adversarial=True, device=device)
torch.save(main_model.state_dict(), "models/dam_cma_final.pth")

print("\n--- Training Simple Fusion Baseline ---")
train_model(fusion_baseline, train_loader, epochs=5, device=device)

print("\n--- Training Text-Only Baseline ---")
train_model(text_baseline, train_loader, epochs=5, device=device)

print("\n--- Training Image-Only Baseline ---")
train_model(image_baseline, train_loader, epochs=5, device=device)

print("\n✅ ALL MODELS TRAINED SUCCESSFULLY!")

## 4. Download Weights
Run this cell to zip and download your trained model weights for use in your local UI.

In [ ]:
from google.colab import files
!zip -r trained_weights.zip models/
files.download('trained_weights.zip')